[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day32-kv-cache-quantization.ipynb)

# Day 32 — KV Cache Quantization

**Tags:** `CPU-OK` · **Time:** 30–60 min · **Prereqs:** Day 8 (KV byte math), Day 29 (affine quantization)

Weights are frozen — the KV cache is a live stream. Today: per-token asymmetric
int8 quantization (the streaming answer), the KIVI asymmetry (K per-channel,
V per-token), and the long-context money table. All CPU.


In [ ]:
# Cell 1: one-time installs. CPU-only day — torch, numpy, matplotlib ship with Colab.
# No extra installs needed.
import torch, math
print("torch", torch.__version__, "| device: cpu (all cells run on CPU)")
# Expected: torch 2.x.x | device: cpu (all cells run on CPU)


## Part 1 — Per-token asymmetric int8: quantize, dequantize, verify the error bound

One (scale, zero-point) per token per head, computed from the token's own
min/max. Roundtrip error must stay ≤ scale/2 — the affine-quantization
guarantee from Day 29, now applied online.


In [ ]:
# Cell 2: per-token asymmetric int8 quantize / dequantize
def quantize_per_token(x):
    # x: (..., dim) -> one scale + zero-point per row (row = one token, one head)
    mn = x.amin(dim=-1, keepdim=True)
    mx = x.amax(dim=-1, keepdim=True)
    scale = torch.clamp((mx - mn) / 255.0, min=1e-8)
    q = torch.round((x - mn) / scale).clamp(0, 255).to(torch.uint8)
    return q, scale.squeeze(-1), mn.squeeze(-1)

def dequantize(q, scale, mn):
    return q.float() * scale.unsqueeze(-1) + mn.unsqueeze(-1)

torch.manual_seed(32)
x = torch.randn(2, 32) * 0.5
q, s, z = quantize_per_token(x)
err = (dequantize(q, s, z) - x).abs().max().item()
bound = (s / 2).max().item()
print(f"max err {err:.4f} <= scale/2 {bound:.4f}: {err <= bound + 1e-6}")
# Expected: max err ~0.01 <= scale/2 ~0.012: True


## Part 2 — Attention with a quantized cache: how much damage?

Toy cache: 512 tokens, 2 KV heads, head_dim 32. Key channel 0 is an outlier
(-50, like real salient channels); value token 0 is an attention sink (20x).
Run causal attention twice — fp16 cache vs per-token int8 cache (dequantized
before the matmul, as a fused kernel would do in registers).


In [ ]:
# Cell 3: fp16 vs int8-KV causal attention
seq, nh, d = 512, 2, 32
torch.manual_seed(32)
K = torch.randn(seq, nh, d) * 0.5
K[:, :, 0] = -50.0 + torch.randn(seq, nh) * 0.5   # outlier CHANNEL in K
V = torch.randn(seq, nh, d) * 0.5
V[0] = 20.0 * torch.randn(nh, d)                  # outlier TOKEN in V (sink)
Q = torch.randn(seq, nh, d) * 0.5

def causal_attn(Q, K, V):
    scores = torch.einsum('qhd,khd->hqk', Q, K) / math.sqrt(d)
    scores = scores.masked_fill(torch.triu(torch.ones(seq, seq), 1).bool(), float('-inf'))
    P = torch.softmax(scores, dim=-1)
    return P, torch.einsum('hqk,khd->qhd', P, V)

P_fp, out_fp = causal_attn(Q, K, V)

Kq, Ks, Kz = quantize_per_token(K.reshape(-1, d))   # per token per head
Vq, Vs, Vz = quantize_per_token(V.reshape(-1, d))
K_dq = dequantize(Kq, Ks, Kz).reshape(seq, nh, d)
V_dq = dequantize(Vq, Vs, Vz).reshape(seq, nh, d)
P_q, out_q = causal_attn(Q, K_dq, V_dq)

dscore = (P_fp - P_q).abs().max().item()
cos = torch.nn.functional.cosine_similarity(out_fp.flatten(), out_q.flatten(), dim=0).item()
agree = (P_fp.argmax(-1) == P_q.argmax(-1)).float().mean().item()
print(f"max |dScore| = {dscore:.3f}")
print(f"cosine sim of outputs = {cos:.5f}  (target > 0.999)")
print(f"top-1 attention position agreement = {agree:.2%}")
# Expected (approx — your torch run lands nearby):
# max |dScore| ~ 0.02
# cosine sim of outputs ~ 0.99990  (target > 0.999)
# top-1 attention position agreement ~ 84% (flips happen among near-tied scores;
#   outputs stay near-identical anyway)


## Part 3 — KIVI demo: why K wants per-channel and V wants per-token

The outlier channel (-50) stretches the per-token scale and crushes the other
31 channels. Quantize K per-channel instead (one scale per channel over tokens
— the offline ideal KIVI approximates with grouped quantization + a fp16
residual window) and compare the error on the *non-outlier* channels.


In [ ]:
# Cell 4: per-channel vs per-token quantization of K
mn_c = K.amin(dim=0, keepdim=True)          # per (head, channel) over tokens
mx_c = K.amax(dim=0, keepdim=True)
sc_c = torch.clamp((mx_c - mn_c) / 255.0, min=1e-8)
K_ch = (torch.round((K - mn_c) / sc_c).clamp(0, 255).float() * sc_c + mn_c)

err_ch  = (K_ch - K)[:, :, 1:].abs().mean().item()   # non-outlier channels
err_tok = (K_dq - K)[:, :, 1:].abs().mean().item()
print(f"per-channel K err (normal ch): {err_ch:.4f}")
print(f"per-token   K err (normal ch): {err_tok:.4f}")
print(f"per-channel is ~{err_tok/err_ch:.0f}x more accurate where it matters")
print("KIVI rule: K per-channel (outlier channels), V per-token (outlier/sink tokens).")
# Expected (approx):
# per-channel K err (normal ch): ~0.003
# per-token   K err (normal ch): ~0.05
# per-channel is ~16x more accurate where it matters (expect 10-25x on your run)


## Part 4 — The money table: KV gigabytes for Llama-3.1-8B

KV bytes/token = 2 (K,V) x 32 layers x 8 KV heads x 128 head_dim x dtype bytes.
fp16 = 128 KB/token, int8 = 64 KB, int4 = 32 KB. Which (seq, batch, dtype)
fits on one 80 GB GPU next to 16 GB of fp16 weights?


In [ ]:
# Cell 5: KV memory table + fit check vs an 80 GB GPU
def kv_gb(seq_len, batch, dtype_bytes, layers=32, kv_heads=8, head_dim=128):
    return 2 * layers * seq_len * kv_heads * head_dim * dtype_bytes * batch / 2**30  # GiB

print(f"{'seq':>7} | {'batch 1 (fp16/int8/int4) GB':>32} | {'batch 16 (fp16/int8/int4) GB':>33}")
for sl in (8192, 32768, 131072):
    b1  = [kv_gb(sl, 1, b) for b in (2, 1, 0.5)]
    b16 = [kv_gb(sl, 16, b) for b in (2, 1, 0.5)]
    print(f"{sl:>7} | {b1[0]:6.2f} / {b1[1]:6.2f} / {b1[2]:6.2f}          "
          f"| {b16[0]:6.1f} / {b16[1]:6.1f} / {b16[2]:6.1f}")

print("\nFit on 80 GB with 16 GB fp16 weights, batch 16, 32k ctx:")
for name, db in (("fp16", 2), ("int8", 1), ("int4", 0.5)):
    total = kv_gb(32768, 16, db) + 16
    print(f"  {name:>4}: {total:5.1f} GB total -> {'YES' if total < 80 else 'NO'}")
# Expected: batch-16/32k row = 64.0 / 32.0 / 16.0 GB; only int8 and int4 fit.


## Part 5 — READ-ONLY: vLLM's `--kv-cache-dtype fp8`

No GPU needed — read and note. vLLM exposes `--kv-cache-dtype` (`auto` /
`fp8` / `fp8_e5m2`). The `fp8` setting stores the KV cache in E4M3 with fused
dequantize-inside-attention kernels, buying the 2x memory cut at near-zero
accuracy loss with no code changes. Requirements and gotchas:

1. **Hopper (sm_90) or newer only** — fp8 tensor-core / conversion instructions
   don't exist on Ampere/Ada (A100, T4, 4090). On older GPUs the flag is a no-op.
2. The kernel must dequantize **on the fly in registers** (Day 9 lesson):
   materializing fp16 first would move the same 128 KB/token and keep decode
   exactly as memory-bandwidth-bound as before.
3. KIVI-style 2-bit remains research-grade in most engines (custom kernels per
   GPU) — per-token int8 / fp8 is the production default.

**Write one line in your notes:** why fp8 KV needs no accuracy tuning (E4M3's
±448 range covers KV magnitudes; attention is noise-tolerant — Part 2 proved it).

## Checkpoints (answer cold before moving on)

1. Why per-token scales for KV but per-channel (or finer) for weights?
2. KIVI's asymmetry: K per-___? V per-___? Why?
3. 8B, batch 16, 32k ctx: fp16 vs int8 vs int4 KV in GB?
4. One K channel at -50, rest in [-1,+1]: why does per-token int8 hurt the
   normal channels ~25x more than per-channel would?
5. Why must the attention kernel dequantize on the fly?

**Tomorrow:** Day 33 — FP8 (E4M3 vs E5M2): the format Hopper made the inference
standard, and why DeepSeek-V3 trains *and* infers in FP8 natively.
